# Generate Sample Sensor Data

This notebook creates sample machine sensor readings, similar in structure to the AI4I 2020 Predictive Maintenance dataset. This lets the project run right away without downloading anything.

When the real dataset is available, replace this notebook's output with the real sensor readings. Just keep the same column names.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

## Simulate Machine Readings

Each row is one machine reading: air temperature, process temperature, rotational speed, torque, and tool wear. We build machine failure from a combination of these values, since in real machines a failure usually comes from more than one factor at once.

In [2]:
machine_count = 5000
rows = []

for i in range(machine_count):
    air_temp = np.random.normal(300, 2)
    process_temp = air_temp + np.random.normal(10, 1)
    rotational_speed = np.random.normal(1500, 150)
    torque = np.random.normal(40, 10)
    tool_wear = np.random.uniform(0, 250)

    rows.append({
        "air_temperature_k": round(air_temp, 2),
        "process_temperature_k": round(process_temp, 2),
        "rotational_speed_rpm": round(rotational_speed, 1),
        "torque_nm": round(torque, 2),
        "tool_wear_min": round(tool_wear, 1),
    })

sensor_data = pd.DataFrame(rows)
sensor_data.head()

,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min
0,300.99,310.86,1597.2,55.23,39.0
1,300.56,311.57,1412.9,34.75,45.9
2,300.48,308.57,1241.3,34.38,153.0
3,299.14,308.40,1394.5,18.60,196.3
4,300.14,308.71,1418.3,41.11,151.9


## Build the Failure Label

We turn each reading into a risk score. High tool wear, high torque, low rotational speed, and a big gap between process and air temperature all push the risk up. This mirrors how real machines tend to fail, where wear and strain combine rather than acting alone.

In [3]:
wear_factor = sensor_data["tool_wear_min"] / 250
torque_factor = ((sensor_data["torque_nm"] - 40) / 30).clip(lower=0)
speed_factor = ((1400 - sensor_data["rotational_speed_rpm"]) / 300).clip(lower=0)
heat_factor = ((sensor_data["process_temperature_k"] - sensor_data["air_temperature_k"] - 10) / 5).clip(lower=0)

risk_score = 0.03 + 0.35 * wear_factor + 0.30 * torque_factor + 0.20 * speed_factor + 0.15 * heat_factor
risk_score = risk_score.clip(upper=0.95)

sensor_data["machine_failure"] = (np.random.random(len(sensor_data)) < risk_score).astype(int)
sensor_data["machine_failure"].value_counts(normalize=True)

machine_failure
0    0.729
1    0.271
Name: proportion, dtype: float64

## Save the Dataset

In [4]:
sensor_data.to_csv("../data/raw/sensor_data.csv", index=False)
print("saved sensor_data.csv with", len(sensor_data), "rows")

saved sensor_data.csv with 5000 rows
